# Standard CNN Experiment

This notebook analyzes the trained Standard CNN.

The model is used as the high-capacity baseline for comparison with the lightweight CNN.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch

from src.models.standard_cnn import StandardCNN
from src.models.model_utils import count_parameters, model_size_mb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StandardCNN().to(device)

print("Device:", device)
print("Trainable parameters:", count_parameters(model))
print("Model size:", model_size_mb(model), "MB")


Device: cuda
Trainable parameters: 896906
Model size: 3.4214248657226562 MB


In [2]:
from src.data.dataset import get_dataloaders

_, _, test_loader = get_dataloaders(batch_size=128)

model_path = ROOT / "models" / "standard_cnn" / "best_model.pth"

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=False
)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 99.05%


## Standard CNN Result

The trained Standard CNN achieved approximately **99.05% test accuracy**.

It contains approximately **896,906 trainable parameters**.
